# 🔍 Проверка Income Statement на всех этапах

**Цель**: Проверить, что Income Statement правильно собирается на всех этапах:
1. Официальная отчетность
2. RAW БД (после загрузки из Edgar CSV)
3. CANONICAL БД (после normalize_to_canonical)
4. **Иерархическая каноническая структура (CanonicalFinancialStatements)** ⭐
5. Препроцессинг (загрузка исторических данных)
6. Движок модели (ThreeStatementModel)

**Основные проверки**:
- Revenue - COGS = Gross Profit
- Gross Profit - SGA - Total DA = EBITDA (или EBIT)
- EBIT + Interest Income - Interest Expense = EBT
- EBT - Tax Expense = Net Income


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("../../..").resolve()
COMPANY = "us_steel"
croot = ROOT / "companies" / COMPANY

sys.path.insert(0, str(ROOT))

from engine.database.data_mart import get_data_mart
from engine.model.preprocess import ModelPreprocessor
from engine.model3.core import ThreeStatementModel
from engine.model3.io import load_inputs

print(f"✅ Импорты выполнены")
print(f"   ROOT: {ROOT}")
print(f"   COMPANY: {COMPANY}")

## 📊 ШАГ 1: Официальная отчетность (2024)


In [ ]:
# Официальные данные IS 2024 из ноутбука 02_Fix_Canonical_Format_IS.ipynb
official_is_2024 = {
    'revenue': 15_640_000_000,  # Total net sales
    'cogs': 14_060_000_000,
    'gross_profit': 1_580_000_000,  # Revenue - COGS
    'sga': 435_000_000,
    'total_da': 913_000_000,
    'ebitda': 1_145_000_000,  # Gross Profit - SGA (приблизительно)
    'ebit': 240_000_000,  # Earnings before interest and income taxes
    'interest_income': -198_000_000,  # отрицательное = доход (Net interest and other financial benefits)
    'interest_expense': 24_000_000,
    'other_income': -137_000_000,  # Earnings from investees + Net gain from investments
    'other_expenses': 56_000_000,  # Asset impairment + Restructuring + Other financial costs + Other losses
    'ebt': 438_000_000,  # Earnings before income taxes
    'tax_expense': 54_000_000,
    'net_income': 384_000_000,
}

print("="*80)
print("📊 ШАГ 1: ОФИЦИАЛЬНАЯ ОТЧЕТНОСТЬ (2024)")
print("="*80)

# Проверка связок
revenue_official = official_is_2024['revenue']
cogs_official = official_is_2024['cogs']
gross_profit_official = official_is_2024['gross_profit']
sga_official = official_is_2024['sga']
total_da_official = official_is_2024['total_da']
ebit_official = official_is_2024['ebit']
interest_income_official = official_is_2024['interest_income']
interest_expense_official = official_is_2024['interest_expense']
ebt_official = official_is_2024['ebt']
tax_expense_official = official_is_2024['tax_expense']
net_income_official = official_is_2024['net_income']

# Проверка: Revenue - COGS = Gross Profit
gross_profit_calc = revenue_official - cogs_official
diff_gross_profit = gross_profit_official - gross_profit_calc

# Проверка: EBIT + Interest Income - Interest Expense = EBT
ebt_calc = ebit_official + interest_income_official - interest_expense_official
diff_ebt = ebt_official - ebt_calc

# Проверка: EBT - Tax Expense = Net Income
net_income_calc = ebt_official - tax_expense_official
diff_net_income = net_income_official - net_income_calc

print(f"\n📋 Income Statement из официальной отчетности:")
print(f"   Revenue: ${revenue_official:,.0f}")
print(f"   COGS: ${cogs_official:,.0f}")
print(f"   Gross Profit: ${gross_profit_official:,.0f}")
print(f"   SGA: ${sga_official:,.0f}")
print(f"   Total DA: ${total_da_official:,.0f}")
print(f"   EBIT: ${ebit_official:,.0f}")
print(f"   Interest Income: ${interest_income_official:,.0f}")
print(f"   Interest Expense: ${interest_expense_official:,.0f}")
print(f"   EBT: ${ebt_official:,.0f}")
print(f"   Tax Expense: ${tax_expense_official:,.0f}")
print(f"   Net Income: ${net_income_official:,.0f}")

print(f"\n📊 Проверка связок:")
print(f"   Revenue - COGS = Gross Profit: ${revenue_official:,.0f} - ${cogs_official:,.0f} = ${gross_profit_calc:,.0f} (ожидается ${gross_profit_official:,.0f}, diff: ${diff_gross_profit:,.0f})")
print(f"   EBIT + Int Income - Int Expense = EBT: ${ebit_official:,.0f} + ${interest_income_official:,.0f} - ${interest_expense_official:,.0f} = ${ebt_calc:,.0f} (ожидается ${ebt_official:,.0f}, diff: ${diff_ebt:,.0f})")
print(f"   EBT - Tax Expense = Net Income: ${ebt_official:,.0f} - ${tax_expense_official:,.0f} = ${net_income_calc:,.0f} (ожидается ${net_income_official:,.0f}, diff: ${diff_net_income:,.0f})")

all_ok = abs(diff_gross_profit) < 1e-6 and abs(diff_ebt) < 1e-6 and abs(diff_net_income) < 1e-6
if all_ok:
    print(f"\n✅ Все связки сходятся!")
else:
    print(f"\n⚠️ Есть расхождения в связках!")

## 📊 ШАГ 2: RAW БД (после загрузки из Edgar CSV)


In [ ]:
mart = get_data_mart(ROOT, COMPANY)

# Загружаем RAW данные
is_raw = mart.get_history_income_statement(canonical=False)

print("="*80)
print("📊 ШАГ 2: RAW БД (после загрузки из Edgar CSV)")
print("="*80)

# Получаем значения для 2024 года
year = 2024
year_str = str(year)

def get_raw_value(metric_name):
    """Получает значение метрики из RAW БД"""
    if is_raw.empty:
        return None
    metric_row = is_raw[is_raw['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

# Получаем ключевые метрики
revenue_raw = get_raw_value('revenue') or get_raw_value('net_sales') or get_raw_value('total_net_sales')
cogs_raw = get_raw_value('cogs') or get_raw_value('cost_of_sales')
gross_profit_raw = get_raw_value('gross_profit')
sga_raw = get_raw_value('sga') or get_raw_value('selling_general_and_administrative_expenses')
total_da_raw = get_raw_value('total_da') or get_raw_value('depreciation_amortization')
ebit_raw = get_raw_value('ebit') or get_raw_value('operating_income')
interest_income_raw = get_raw_value('interest_income')
interest_expense_raw = get_raw_value('interest_expense')
ebt_raw = get_raw_value('ebt') or get_raw_value('earnings_before_income_taxes')
tax_expense_raw = get_raw_value('tax_expense') or get_raw_value('income_tax_expense')
net_income_raw = get_raw_value('net_income') or get_raw_value('net_earnings')

# Проверка связок
gross_profit_calc_raw = revenue_raw - cogs_raw if pd.notna(revenue_raw) and pd.notna(cogs_raw) else None
ebt_calc_raw = ebit_raw + interest_income_raw - interest_expense_raw if pd.notna(ebit_raw) and pd.notna(interest_income_raw) and pd.notna(interest_expense_raw) else None
net_income_calc_raw = ebt_raw - tax_expense_raw if pd.notna(ebt_raw) and pd.notna(tax_expense_raw) else None

print(f"\n📋 Income Statement из RAW БД:")
print(f"   Revenue: ${revenue_raw:,.0f}" if pd.notna(revenue_raw) else "   Revenue: N/A")
print(f"   COGS: ${cogs_raw:,.0f}" if pd.notna(cogs_raw) else "   COGS: N/A")
print(f"   Gross Profit: ${gross_profit_raw:,.0f}" if pd.notna(gross_profit_raw) else "   Gross Profit: N/A")
print(f"   SGA: ${sga_raw:,.0f}" if pd.notna(sga_raw) else "   SGA: N/A")
print(f"   Total DA: ${total_da_raw:,.0f}" if pd.notna(total_da_raw) else "   Total DA: N/A")
print(f"   EBIT: ${ebit_raw:,.0f}" if pd.notna(ebit_raw) else "   EBIT: N/A")
print(f"   Interest Income: ${interest_income_raw:,.0f}" if pd.notna(interest_income_raw) else "   Interest Income: N/A")
print(f"   Interest Expense: ${interest_expense_raw:,.0f}" if pd.notna(interest_expense_raw) else "   Interest Expense: N/A")
print(f"   EBT: ${ebt_raw:,.0f}" if pd.notna(ebt_raw) else "   EBT: N/A")
print(f"   Tax Expense: ${tax_expense_raw:,.0f}" if pd.notna(tax_expense_raw) else "   Tax Expense: N/A")
print(f"   Net Income: ${net_income_raw:,.0f}" if pd.notna(net_income_raw) else "   Net Income: N/A")

if pd.notna(gross_profit_calc_raw):
    diff_gross_profit_raw = gross_profit_raw - gross_profit_calc_raw if pd.notna(gross_profit_raw) else None
    if diff_gross_profit_raw is not None:
        print(f"\n📊 Проверка связок:")
        print(f"   Revenue - COGS = Gross Profit: diff = ${diff_gross_profit_raw:,.0f}")
        if abs(diff_gross_profit_raw) < 1e-6:
            print(f"   ✅ Связка Revenue - COGS = Gross Profit сходится!")
        else:
            print(f"   ⚠️ Связка Revenue - COGS = Gross Profit НЕ сходится!")

if pd.notna(ebt_calc_raw):
    diff_ebt_raw = ebt_raw - ebt_calc_raw if pd.notna(ebt_raw) else None
    if diff_ebt_raw is not None:
        print(f"   EBIT + Int Income - Int Expense = EBT: diff = ${diff_ebt_raw:,.0f}")
        if abs(diff_ebt_raw) < 1e-6:
            print(f"   ✅ Связка EBIT + Int Income - Int Expense = EBT сходится!")
        else:
            print(f"   ⚠️ Связка EBIT + Int Income - Int Expense = EBT НЕ сходится!")

if pd.notna(net_income_calc_raw):
    diff_net_income_raw = net_income_raw - net_income_calc_raw if pd.notna(net_income_raw) else None
    if diff_net_income_raw is not None:
        print(f"   EBT - Tax Expense = Net Income: diff = ${diff_net_income_raw:,.0f}")
        if abs(diff_net_income_raw) < 1e-6:
            print(f"   ✅ Связка EBT - Tax Expense = Net Income сходится!")
        else:
            print(f"   ⚠️ Связка EBT - Tax Expense = Net Income НЕ сходится!")

# Сравнение с официальной отчетностью
if pd.notna(revenue_raw):
    diff_vs_official = revenue_raw - revenue_official
    print(f"\n📊 Сравнение с официальной отчетностью:")
    print(f"   Revenue: ${revenue_raw:,.0f} vs ${revenue_official:,.0f} (diff: ${diff_vs_official:,.0f})")


## 📊 ШАГ 3: CANONICAL БД (после normalize_to_canonical)


In [ ]:
# Загружаем CANONICAL данные
is_canonical = mart.get_history_income_statement(canonical=True)

print("="*80)
print("📊 ШАГ 3: CANONICAL БД (после normalize_to_canonical)")
print("="*80)

def get_canonical_value(metric_name):
    """Получает значение метрики из CANONICAL БД"""
    if is_canonical.empty:
        return None
    metric_row = is_canonical[is_canonical['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

# Получаем значения
revenue_canonical = get_canonical_value('revenue')
cogs_canonical = get_canonical_value('cogs')
gross_profit_canonical = get_canonical_value('gross_profit')
sga_canonical = get_canonical_value('sga')
total_da_canonical = get_canonical_value('total_da')
ebit_canonical = get_canonical_value('ebit')
interest_income_canonical = get_canonical_value('interest_income')
interest_expense_canonical = get_canonical_value('interest_expense')
ebt_canonical = get_canonical_value('ebt')
tax_expense_canonical = get_canonical_value('tax_expense')
net_income_canonical = get_canonical_value('net_income')

if pd.isna(revenue_canonical) or pd.isna(cogs_canonical):
    print("⚠️ Не все ключевые метрики найдены в CANONICAL БД")
else:
    # Проверка связок
    gross_profit_calc_canonical = revenue_canonical - cogs_canonical
    ebt_calc_canonical = ebit_canonical + interest_income_canonical - interest_expense_canonical if pd.notna(ebit_canonical) and pd.notna(interest_income_canonical) and pd.notna(interest_expense_canonical) else None
    net_income_calc_canonical = ebt_canonical - tax_expense_canonical if pd.notna(ebt_canonical) and pd.notna(tax_expense_canonical) else None

    print(f"\n📋 Income Statement из CANONICAL БД:")
    print(f"   Revenue: ${revenue_canonical:,.0f}")
    print(f"   COGS: ${cogs_canonical:,.0f}")
    print(f"   Gross Profit: ${gross_profit_canonical:,.0f}" if pd.notna(gross_profit_canonical) else "   Gross Profit: N/A")
    print(f"   SGA: ${sga_canonical:,.0f}" if pd.notna(sga_canonical) else "   SGA: N/A")
    print(f"   Total DA: ${total_da_canonical:,.0f}" if pd.notna(total_da_canonical) else "   Total DA: N/A")
    print(f"   EBIT: ${ebit_canonical:,.0f}" if pd.notna(ebit_canonical) else "   EBIT: N/A")
    print(f"   Interest Income: ${interest_income_canonical:,.0f}" if pd.notna(interest_income_canonical) else "   Interest Income: N/A")
    print(f"   Interest Expense: ${interest_expense_canonical:,.0f}" if pd.notna(interest_expense_canonical) else "   Interest Expense: N/A")
    print(f"   EBT: ${ebt_canonical:,.0f}" if pd.notna(ebt_canonical) else "   EBT: N/A")
    print(f"   Tax Expense: ${tax_expense_canonical:,.0f}" if pd.notna(tax_expense_canonical) else "   Tax Expense: N/A")
    print(f"   Net Income: ${net_income_canonical:,.0f}" if pd.notna(net_income_canonical) else "   Net Income: N/A")

    print(f"\n📊 Проверка связок:")
    diff_gross_profit_canonical = gross_profit_canonical - gross_profit_calc_canonical if pd.notna(gross_profit_canonical) else None
    if diff_gross_profit_canonical is not None:
        print(f"   Revenue - COGS = Gross Profit: diff = ${diff_gross_profit_canonical:,.0f}")
        if abs(diff_gross_profit_canonical) < 1e-6:
            print(f"   ✅ Связка Revenue - COGS = Gross Profit сходится!")
        else:
            print(f"   ⚠️ Связка Revenue - COGS = Gross Profit НЕ сходится!")

    if pd.notna(ebt_calc_canonical):
        diff_ebt_canonical = ebt_canonical - ebt_calc_canonical if pd.notna(ebt_canonical) else None
        if diff_ebt_canonical is not None:
            print(f"   EBIT + Int Income - Int Expense = EBT: diff = ${diff_ebt_canonical:,.0f}")
            if abs(diff_ebt_canonical) < 1e-6:
                print(f"   ✅ Связка EBIT + Int Income - Int Expense = EBT сходится!")
            else:
                print(f"   ⚠️ Связка EBIT + Int Income - Int Expense = EBT НЕ сходится!")

    if pd.notna(net_income_calc_canonical):
        diff_net_income_canonical = net_income_canonical - net_income_calc_canonical if pd.notna(net_income_canonical) else None
        if diff_net_income_canonical is not None:
            print(f"   EBT - Tax Expense = Net Income: diff = ${diff_net_income_canonical:,.0f}")
            if abs(diff_net_income_canonical) < 1e-6:
                print(f"   ✅ Связка EBT - Tax Expense = Net Income сходится!")
            else:
                print(f"   ⚠️ Связка EBT - Tax Expense = Net Income НЕ сходится!")

    # Сравнение с RAW БД
    print(f"\n📊 Сравнение с RAW БД:")
    if pd.notna(revenue_raw):
        print(f"   Revenue: ${revenue_canonical:,.0f} vs ${revenue_raw:,.0f} (diff: ${revenue_canonical - revenue_raw:,.0f})")
    if pd.notna(cogs_raw):
        print(f"   COGS: ${cogs_canonical:,.0f} vs ${cogs_raw:,.0f} (diff: ${cogs_canonical - cogs_raw:,.0f})")
    if pd.notna(net_income_raw):
        print(f"   Net Income: ${net_income_canonical:,.0f} vs ${net_income_raw:,.0f} (diff: ${net_income_canonical - net_income_raw:,.0f})")


## 📊 ШАГ 3.5: Иерархическая каноническая структура (CanonicalFinancialStatements) ⭐


In [ ]:
# Загружаем иерархическую каноническую структуру
print("="*80)
print("📊 ШАГ 3.5: ИЕРАРХИЧЕСКАЯ КАНОНИЧЕСКАЯ СТРУКТУРА (CanonicalFinancialStatements)")
print("="*80)

canonical_hierarchical = mart.get_canonical_financial_statements(year=year)

if canonical_hierarchical is None:
    print("⚠️ Иерархическая каноническая структура недоступна (адаптеры не найдены или данные отсутствуют)")
    revenue_hierarchical = None
    cogs_hierarchical = None
    gross_profit_hierarchical = None
    ebit_hierarchical = None
    ebt_hierarchical = None
    net_income_hierarchical = None
else:
    # Получаем значения из иерархической структуры
    is_hierarchical = canonical_hierarchical.income_statement
    
    # Получаем ключевые метрики
    revenue_hierarchical = is_hierarchical.revenue_total
    cogs_hierarchical = is_hierarchical.cogs_total
    gross_profit_hierarchical = is_hierarchical.gross_profit
    sga_hierarchical = is_hierarchical.sga_total
    total_da_hierarchical = is_hierarchical.total_depr_amort
    ebit_hierarchical = is_hierarchical.ebit
    interest_income_hierarchical = is_hierarchical.interest_income
    interest_expense_hierarchical = is_hierarchical.interest_expense
    ebt_hierarchical = is_hierarchical.ebt
    tax_expense_hierarchical = is_hierarchical.total_tax_expense
    net_income_hierarchical = is_hierarchical.net_income
    
    # Проверка связок
    gross_profit_calc_hierarchical = revenue_hierarchical - cogs_hierarchical if pd.notna(revenue_hierarchical) and pd.notna(cogs_hierarchical) else None
    ebt_calc_hierarchical = ebit_hierarchical + interest_income_hierarchical - interest_expense_hierarchical if pd.notna(ebit_hierarchical) and pd.notna(interest_income_hierarchical) and pd.notna(interest_expense_hierarchical) else None
    net_income_calc_hierarchical = ebt_hierarchical - tax_expense_hierarchical if pd.notna(ebt_hierarchical) and pd.notna(tax_expense_hierarchical) else None
    
    print(f"\n📋 Income Statement из иерархической канонической структуры:")
    print(f"   Revenue: ${revenue_hierarchical:,.0f}" if pd.notna(revenue_hierarchical) else "   Revenue: N/A")
    print(f"   COGS: ${cogs_hierarchical:,.0f}" if pd.notna(cogs_hierarchical) else "   COGS: N/A")
    print(f"   Gross Profit: ${gross_profit_hierarchical:,.0f}" if pd.notna(gross_profit_hierarchical) else "   Gross Profit: N/A")
    print(f"   SGA: ${sga_hierarchical:,.0f}" if pd.notna(sga_hierarchical) else "   SGA: N/A")
    print(f"   Total DA: ${total_da_hierarchical:,.0f}" if pd.notna(total_da_hierarchical) else "   Total DA: N/A")
    print(f"   EBIT: ${ebit_hierarchical:,.0f}" if pd.notna(ebit_hierarchical) else "   EBIT: N/A")
    print(f"   Interest Income: ${interest_income_hierarchical:,.0f}" if pd.notna(interest_income_hierarchical) else "   Interest Income: N/A")
    print(f"   Interest Expense: ${interest_expense_hierarchical:,.0f}" if pd.notna(interest_expense_hierarchical) else "   Interest Expense: N/A")
    print(f"   EBT: ${ebt_hierarchical:,.0f}" if pd.notna(ebt_hierarchical) else "   EBT: N/A")
    print(f"   Tax Expense: ${tax_expense_hierarchical:,.0f}" if pd.notna(tax_expense_hierarchical) else "   Tax Expense: N/A")
    print(f"   Net Income: ${net_income_hierarchical:,.0f}" if pd.notna(net_income_hierarchical) else "   Net Income: N/A")
    
    if pd.notna(gross_profit_calc_hierarchical):
        diff_gross_profit_hierarchical = gross_profit_hierarchical - gross_profit_calc_hierarchical if pd.notna(gross_profit_hierarchical) else None
        if diff_gross_profit_hierarchical is not None:
            print(f"\n📊 Проверка связок:")
            print(f"   Revenue - COGS = Gross Profit: diff = ${diff_gross_profit_hierarchical:,.0f}")
            if abs(diff_gross_profit_hierarchical) < 1e-6:
                print(f"   ✅ Связка Revenue - COGS = Gross Profit сходится!")
            else:
                print(f"   ⚠️ Связка Revenue - COGS = Gross Profit НЕ сходится!")
    
    if pd.notna(ebt_calc_hierarchical):
        diff_ebt_hierarchical = ebt_hierarchical - ebt_calc_hierarchical if pd.notna(ebt_hierarchical) else None
        if diff_ebt_hierarchical is not None:
            print(f"   EBIT + Int Income - Int Expense = EBT: diff = ${diff_ebt_hierarchical:,.0f}")
            if abs(diff_ebt_hierarchical) < 1e-6:
                print(f"   ✅ Связка EBIT + Int Income - Int Expense = EBT сходится!")
            else:
                print(f"   ⚠️ Связка EBIT + Int Income - Int Expense = EBT НЕ сходится!")
    
    if pd.notna(net_income_calc_hierarchical):
        diff_net_income_hierarchical = net_income_hierarchical - net_income_calc_hierarchical if pd.notna(net_income_hierarchical) else None
        if diff_net_income_hierarchical is not None:
            print(f"   EBT - Tax Expense = Net Income: diff = ${diff_net_income_hierarchical:,.0f}")
            if abs(diff_net_income_hierarchical) < 1e-6:
                print(f"   ✅ Связка EBT - Tax Expense = Net Income сходится!")
            else:
                print(f"   ⚠️ Связка EBT - Tax Expense = Net Income НЕ сходится!")
    
    # Сравнение с CANONICAL БД (плоский формат)
    if pd.notna(revenue_canonical):
        print(f"\n📊 Сравнение с CANONICAL БД (плоский формат):")
        print(f"   Revenue: ${revenue_hierarchical:,.0f} vs ${revenue_canonical:,.0f} (diff: ${revenue_hierarchical - revenue_canonical:,.0f})")
        if pd.notna(cogs_canonical):
            print(f"   COGS: ${cogs_hierarchical:,.0f} vs ${cogs_canonical:,.0f} (diff: ${cogs_hierarchical - cogs_canonical:,.0f})")
        if pd.notna(net_income_canonical):
            print(f"   Net Income: ${net_income_hierarchical:,.0f} vs ${net_income_canonical:,.0f} (diff: ${net_income_hierarchical - net_income_canonical:,.0f})")
    
    # Проверка детализации Revenue (если доступна)
    if is_hierarchical.revenue_product is not None or is_hierarchical.revenue_service is not None:
        revenue_product = is_hierarchical.revenue_product or 0
        revenue_service = is_hierarchical.revenue_service or 0
        revenue_other = is_hierarchical.revenue_other or 0
        revenue_total_breakdown = revenue_product + revenue_service + revenue_other
        print(f"\n💡 Детализация Revenue:")
        print(f"   Product Revenue: ${revenue_product:,.0f}")
        print(f"   Service Revenue: ${revenue_service:,.0f}")
        print(f"   Other Revenue: ${revenue_other:,.0f}")
        print(f"   Total (breakdown): ${revenue_total_breakdown:,.0f} vs Total: ${revenue_hierarchical:,.0f}")
    
    # Проверка детализации налогов (если доступна)
    if is_hierarchical.current_tax_expense is not None or is_hierarchical.deferred_tax_expense is not None:
        current_tax = is_hierarchical.current_tax_expense or 0
        deferred_tax = is_hierarchical.deferred_tax_expense or 0
        total_tax_breakdown = current_tax + deferred_tax
        print(f"\n💡 Детализация налогов:")
        print(f"   Current Tax Expense: ${current_tax:,.0f}")
        print(f"   Deferred Tax Expense: ${deferred_tax:,.0f}")
        print(f"   Total Tax (breakdown): ${total_tax_breakdown:,.0f} vs Total: ${tax_expense_hierarchical:,.0f}")


для 

In [ ]:
# Загружаем исторические данные через препроцессинг
preprocessor = ModelPreprocessor(ROOT, COMPANY)
hist_is = preprocessor.mart.get_history_income_statement(canonical=True)

print("="*80)
print("📊 ШАГ 4: ПРЕПРОЦЕССИНГ (загрузка исторических данных)")
print("="*80)

def get_hist_value(metric_name):
    """Получает значение метрики из исторических данных"""
    if hist_is.empty:
        return None
    metric_row = hist_is[hist_is['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

# Получаем значения
revenue_preprocess = get_hist_value('revenue')
cogs_preprocess = get_hist_value('cogs')
gross_profit_preprocess = get_hist_value('gross_profit')
sga_preprocess = get_hist_value('sga')
total_da_preprocess = get_hist_value('total_da')
ebit_preprocess = get_hist_value('ebit')
interest_income_preprocess = get_hist_value('interest_income')
interest_expense_preprocess = get_hist_value('interest_expense')
ebt_preprocess = get_hist_value('ebt')
tax_expense_preprocess = get_hist_value('tax_expense')
net_income_preprocess = get_hist_value('net_income')

if pd.isna(revenue_preprocess) or pd.isna(cogs_preprocess):
    print("⚠️ Не все ключевые метрики найдены в исторических данных")
else:
    # Проверка связок
    gross_profit_calc_preprocess = revenue_preprocess - cogs_preprocess
    ebt_calc_preprocess = ebit_preprocess + interest_income_preprocess - interest_expense_preprocess if pd.notna(ebit_preprocess) and pd.notna(interest_income_preprocess) and pd.notna(interest_expense_preprocess) else None
    net_income_calc_preprocess = ebt_preprocess - tax_expense_preprocess if pd.notna(ebt_preprocess) and pd.notna(tax_expense_preprocess) else None

    print(f"\n📋 Income Statement из исторических данных:")
    print(f"   Revenue: ${revenue_preprocess:,.0f}")
    print(f"   COGS: ${cogs_preprocess:,.0f}")
    print(f"   Gross Profit: ${gross_profit_preprocess:,.0f}" if pd.notna(gross_profit_preprocess) else "   Gross Profit: N/A")
    print(f"   SGA: ${sga_preprocess:,.0f}" if pd.notna(sga_preprocess) else "   SGA: N/A")
    print(f"   Total DA: ${total_da_preprocess:,.0f}" if pd.notna(total_da_preprocess) else "   Total DA: N/A")
    print(f"   EBIT: ${ebit_preprocess:,.0f}" if pd.notna(ebit_preprocess) else "   EBIT: N/A")
    print(f"   Interest Income: ${interest_income_preprocess:,.0f}" if pd.notna(interest_income_preprocess) else "   Interest Income: N/A")
    print(f"   Interest Expense: ${interest_expense_preprocess:,.0f}" if pd.notna(interest_expense_preprocess) else "   Interest Expense: N/A")
    print(f"   EBT: ${ebt_preprocess:,.0f}" if pd.notna(ebt_preprocess) else "   EBT: N/A")
    print(f"   Tax Expense: ${tax_expense_preprocess:,.0f}" if pd.notna(tax_expense_preprocess) else "   Tax Expense: N/A")
    print(f"   Net Income: ${net_income_preprocess:,.0f}" if pd.notna(net_income_preprocess) else "   Net Income: N/A")

    print(f"\n📊 Проверка связок:")
    diff_gross_profit_preprocess = gross_profit_preprocess - gross_profit_calc_preprocess if pd.notna(gross_profit_preprocess) else None
    if diff_gross_profit_preprocess is not None:
        print(f"   Revenue - COGS = Gross Profit: diff = ${diff_gross_profit_preprocess:,.0f}")
        if abs(diff_gross_profit_preprocess) < 1e-6:
            print(f"   ✅ Связка Revenue - COGS = Gross Profit сходится!")
        else:
            print(f"   ⚠️ Связка Revenue - COGS = Gross Profit НЕ сходится!")

    if pd.notna(ebt_calc_preprocess):
        diff_ebt_preprocess = ebt_preprocess - ebt_calc_preprocess if pd.notna(ebt_preprocess) else None
        if diff_ebt_preprocess is not None:
            print(f"   EBIT + Int Income - Int Expense = EBT: diff = ${diff_ebt_preprocess:,.0f}")
            if abs(diff_ebt_preprocess) < 1e-6:
                print(f"   ✅ Связка EBIT + Int Income - Int Expense = EBT сходится!")
            else:
                print(f"   ⚠️ Связка EBIT + Int Income - Int Expense = EBT НЕ сходится!")

    if pd.notna(net_income_calc_preprocess):
        diff_net_income_preprocess = net_income_preprocess - net_income_calc_preprocess if pd.notna(net_income_preprocess) else None
        if diff_net_income_preprocess is not None:
            print(f"   EBT - Tax Expense = Net Income: diff = ${diff_net_income_preprocess:,.0f}")
            if abs(diff_net_income_preprocess) < 1e-6:
                print(f"   ✅ Связка EBT - Tax Expense = Net Income сходится!")
            else:
                print(f"   ⚠️ Связка EBT - Tax Expense = Net Income НЕ сходится!")

    # Сравнение с CANONICAL БД
    if pd.notna(revenue_canonical):
        print(f"\n📊 Сравнение с CANONICAL БД:")
        print(f"   Revenue: ${revenue_preprocess:,.0f} vs ${revenue_canonical:,.0f} (diff: ${revenue_preprocess - revenue_canonical:,.0f})")
        print(f"   COGS: ${cogs_preprocess:,.0f} vs ${cogs_canonical:,.0f} (diff: ${cogs_preprocess - cogs_canonical:,.0f})")
        print(f"   Net Income: ${net_income_preprocess:,.0f} vs ${net_income_canonical:,.0f} (diff: ${net_income_preprocess - net_income_canonical:,.0f})")


## 📊 ШАГ 5: Движок модели (ThreeStatementModel)


In [ ]:
# Загружаем модельhist_state, history, drivers, canonical = load_inputs(root=ROOT, company=COMPANY)model = ThreeStatementModel(**inputs)model.build()print("="*80)print("📊 ШАГ 5: ДВИЖОК МОДЕЛИ (ThreeStatementModel)")print("="*80)# Получаем Income Statement для 2024 года (последний исторический год)is_2024 = model.IS[2024]# Получаем значения из моделиrevenue_model = is_2024.revenuecogs_model = is_2024.cogsgross_profit_model = is_2024.gross_profitsga_model = is_2024.sgatotal_da_model = is_2024.total_daebit_model = is_2024.ebitinterest_income_model = is_2024.interest_incomeinterest_expense_model = is_2024.interest_expenseebt_model = is_2024.ebttax_expense_model = is_2024.tax_expensenet_income_model = is_2024.net_income# Проверка связокgross_profit_calc_model = revenue_model - cogs_modelebt_calc_model = ebit_model + interest_income_model - interest_expense_modelnet_income_calc_model = ebt_model - tax_expense_modelprint(f"\n📋 Income Statement из модели (2024 год):")print(f"   Revenue: ${revenue_model:,.0f}")print(f"   COGS: ${cogs_model:,.0f}")print(f"   Gross Profit: ${gross_profit_model:,.0f}")print(f"   SGA: ${sga_model:,.0f}")print(f"   Total DA: ${total_da_model:,.0f}")print(f"   EBIT: ${ebit_model:,.0f}")print(f"   Interest Income: ${interest_income_model:,.0f}")print(f"   Interest Expense: ${interest_expense_model:,.0f}")print(f"   EBT: ${ebt_model:,.0f}")print(f"   Tax Expense: ${tax_expense_model:,.0f}")print(f"   Net Income: ${net_income_model:,.0f}")print(f"\n📊 Проверка связок:")diff_gross_profit_model = gross_profit_model - gross_profit_calc_modeldiff_ebt_model = ebt_model - ebt_calc_modeldiff_net_income_model = net_income_model - net_income_calc_modelprint(f"   Revenue - COGS = Gross Profit: diff = ${diff_gross_profit_model:,.0f}")if abs(diff_gross_profit_model) < 1e-6:    print(f"   ✅ Связка Revenue - COGS = Gross Profit сходится!")else:    print(f"   ⚠️ Связка Revenue - COGS = Gross Profit НЕ сходится!")print(f"   EBIT + Int Income - Int Expense = EBT: diff = ${diff_ebt_model:,.0f}")if abs(diff_ebt_model) < 1e-6:    print(f"   ✅ Связка EBIT + Int Income - Int Expense = EBT сходится!")else:    print(f"   ⚠️ Связка EBIT + Int Income - Int Expense = EBT НЕ сходится!")print(f"   EBT - Tax Expense = Net Income: diff = ${diff_net_income_model:,.0f}")if abs(diff_net_income_model) < 1e-6:    print(f"   ✅ Связка EBT - Tax Expense = Net Income сходится!")else:    print(f"   ⚠️ Связка EBT - Tax Expense = Net Income НЕ сходится!")# Сравнение с препроцессингомif pd.notna(revenue_preprocess):    print(f"\n📊 Сравнение с препроцессингом:")    print(f"   Revenue: ${revenue_model:,.0f} vs ${revenue_preprocess:,.0f} (diff: ${revenue_model - revenue_preprocess:,.0f})")    print(f"   COGS: ${cogs_model:,.0f} vs ${cogs_preprocess:,.0f} (diff: ${cogs_model - cogs_preprocess:,.0f})")    print(f"   Net Income: ${net_income_model:,.0f} vs ${net_income_preprocess:,.0f} (diff: ${net_income_model - net_income_preprocess:,.0f})")

## 📊 ИТОГОВАЯ СВОДКА


In [ ]:
from IPython.display import display, Markdown

print("="*80)
print("📊 ИТОГОВАЯ СВОДКА ПРОВЕРКИ INCOME STATEMENT")
print("="*80)

summary_data = []

# Официальная отчетность
summary_data.append({
    'Этап': 'Официальная отчетность',
    'Revenue': f"${revenue_official:,.0f}",
    'COGS': f"${cogs_official:,.0f}",
    'Gross Profit': f"${gross_profit_official:,.0f}",
    'EBIT': f"${ebit_official:,.0f}",
    'EBT': f"${ebt_official:,.0f}",
    'Net Income': f"${net_income_official:,.0f}",
    'Статус': '✅' if all_ok else '⚠️'
})

# RAW БД
if pd.notna(revenue_raw):
    summary_data.append({
        'Этап': 'RAW БД',
        'Revenue': f"${revenue_raw:,.0f}",
        'COGS': f"${cogs_raw:,.0f}" if pd.notna(cogs_raw) else 'N/A',
        'Gross Profit': f"${gross_profit_raw:,.0f}" if pd.notna(gross_profit_raw) else 'N/A',
        'EBIT': f"${ebit_raw:,.0f}" if pd.notna(ebit_raw) else 'N/A',
        'EBT': f"${ebt_raw:,.0f}" if pd.notna(ebt_raw) else 'N/A',
        'Net Income': f"${net_income_raw:,.0f}" if pd.notna(net_income_raw) else 'N/A',
        'Статус': '✅'
    })

# CANONICAL БД
if pd.notna(revenue_canonical):
    summary_data.append({
        'Этап': 'CANONICAL БД',
        'Revenue': f"${revenue_canonical:,.0f}",
        'COGS': f"${cogs_canonical:,.0f}",
        'Gross Profit': f"${gross_profit_canonical:,.0f}" if pd.notna(gross_profit_canonical) else 'N/A',
        'EBIT': f"${ebit_canonical:,.0f}" if pd.notna(ebit_canonical) else 'N/A',
        'EBT': f"${ebt_canonical:,.0f}" if pd.notna(ebt_canonical) else 'N/A',
        'Net Income': f"${net_income_canonical:,.0f}" if pd.notna(net_income_canonical) else 'N/A',
        'Статус': '✅'
    })

# Иерархическая каноническая структура
if revenue_hierarchical is not None:
    summary_data.append({
        'Этап': 'Иерархическая каноническая структура ⭐',
        'Revenue': f"${revenue_hierarchical:,.0f}",
        'COGS': f"${cogs_hierarchical:,.0f}" if pd.notna(cogs_hierarchical) else 'N/A',
        'Gross Profit': f"${gross_profit_hierarchical:,.0f}" if pd.notna(gross_profit_hierarchical) else 'N/A',
        'EBIT': f"${ebit_hierarchical:,.0f}" if pd.notna(ebit_hierarchical) else 'N/A',
        'EBT': f"${ebt_hierarchical:,.0f}" if pd.notna(ebt_hierarchical) else 'N/A',
        'Net Income': f"${net_income_hierarchical:,.0f}" if pd.notna(net_income_hierarchical) else 'N/A',
        'Статус': '✅' if (pd.notna(gross_profit_calc_hierarchical) and abs(diff_gross_profit_hierarchical) < 1e-6 if diff_gross_profit_hierarchical is not None else False) and (pd.notna(ebt_calc_hierarchical) and abs(diff_ebt_hierarchical) < 1e-6 if diff_ebt_hierarchical is not None else False) and (pd.notna(net_income_calc_hierarchical) and abs(diff_net_income_hierarchical) < 1e-6 if diff_net_income_hierarchical is not None else False) else '⚠️'
    })

# Препроцессинг
if pd.notna(revenue_preprocess):
    summary_data.append({
        'Этап': 'Препроцессинг',
        'Revenue': f"${revenue_preprocess:,.0f}",
        'COGS': f"${cogs_preprocess:,.0f}",
        'Gross Profit': f"${gross_profit_preprocess:,.0f}" if pd.notna(gross_profit_preprocess) else 'N/A',
        'EBIT': f"${ebit_preprocess:,.0f}" if pd.notna(ebit_preprocess) else 'N/A',
        'EBT': f"${ebt_preprocess:,.0f}" if pd.notna(ebt_preprocess) else 'N/A',
        'Net Income': f"${net_income_preprocess:,.0f}" if pd.notna(net_income_preprocess) else 'N/A',
        'Статус': '✅'
    })

# Движок модели
summary_data.append({
    'Этап': 'Движок модели',
    'Revenue': f"${revenue_model:,.0f}",
    'COGS': f"${cogs_model:,.0f}",
    'Gross Profit': f"${gross_profit_model:,.0f}",
    'EBIT': f"${ebit_model:,.0f}",
    'EBT': f"${ebt_model:,.0f}",
    'Net Income': f"${net_income_model:,.0f}",
    'Статус': '✅' if abs(diff_gross_profit_model) < 1e-6 and abs(diff_ebt_model) < 1e-6 and abs(diff_net_income_model) < 1e-6 else '⚠️'
})

summary_df = pd.DataFrame(summary_data)
display(Markdown("#### 📊 Сводная таблица проверки Income Statement на всех этапах:"))
display(summary_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print("\n" + "="*80)
print("✅ ПРОВЕРКА ЗАВЕРШЕНА")
print("="*80)
print("\n💡 Интерпретация результатов:")
print("   ✅ - Связки сходятся (Revenue - COGS = Gross Profit, EBIT + Int Income - Int Expense = EBT, EBT - Tax = Net Income)")
print("   ⚠️ - Есть расхождения в связках")
print("\n   Если связки не сходятся на каком-то этапе, проверьте:")
print("   - Правильность маппинга метрик")
print("   - Правильность суммирования статей (combine_from)")
print("   - Потерю данных при преобразовании")
print("   - Логику препроцессинга или движка")

mart.close()
